In [1]:
!pip install -U bitsandbytes>=0.46.1
# temp fix

zsh:1: 0.46.1 not found


In [ ]:
import os
import sys
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Configurazione del percorso e Cache (Colab o Locale)
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    
    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    
    # Impostiamo HF_HOME prima di importare hf_hub o transformers
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    
    print(f"Ambiente Colab e Google Drive configurati. Cache modelli in: {hf_cache_dir}")
except ImportError:
    print("Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.")

/home/zizzis/Code/Politecnico/Large_Language_Models/CLARITY/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.


In [ ]:
# 2. Gestione del Token di Autenticazione
hf_token = None
try:
    from google.colab import userdata
    print("Acquisizione token da Google Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Lettura token dal file .env locale...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("Errore: File .env non trovato.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Autenticazione Hugging Face completata.")
else:
    print("Attenzione: Token HF non trovato. Possibili errori nel download del modello.")

Lettura token dal file .env locale...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Autenticazione Hugging Face completata.


## Evasion-based clarity classification (Classificazione basata sull'evasione)
Si tratta di un task più a grana fine (il livello inferiore della tassonomia) che prevede un approccio in due fasi. Il modello deve prima prevedere l'esatta tecnica di evasione utilizzata scegliendo tra **9 sotto-categorie** specifiche (ad esempio *Implicit*, *Dodging*, *Deflection*, *General*, *Claims ignorance*, ecc.). Una volta individuata la tecnica di evasione, il modello deduce la categoria di chiarezza principale (delle 3 viste sopra) risalendo la gerarchia della tassonomia.

In [ ]:
# 3. Caricamento del Modello (Quantizzazione 4-bit)
model_id = "meta-llama/Llama-3.1-8B-Instruct" 

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Llama 3.1 non ha un pad token di default, usiamo l'eos_token per il fine-tuning
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Inizializzazione configurazione BitsAndBytes per {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Trasferimento dei pesi del modello in VRAM...")

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16  # Corretto da dtype
)

print("Caricamento completato con successo. Sistema pronto per l'inferenza.")

Inizializzazione configurazione BitsAndBytes per meta-llama/Llama-3.1-8B-Instruct...
Trasferimento dei pesi del modello in VRAM...


Fetching 4 files:   0%|          | 0/4 [04:18<?, ?it/s]


In [ ]:
# 4. Preparazione per PEFT (LoRA / DoRA)
print("Preparazione del modello per il training a 4-bit...")
model = prepare_model_for_kbit_training(model)

# Configurazione DoRA basata sui layer di attenzione di Llama
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True # Attiviamo DoRA come suggerito per performance migliori
)

print("Applicazione degli adattatori LoRA/DoRA...")
model = get_peft_model(model, peft_config)

# Stampa un riepilogo per verificare di star addestrando solo una frazione dei parametri
model.print_trainable_parameters()

In [ ]:
# 5. Caricamento e Formattazione per Task 2 (Evasion-based)
from datasets import load_from_disk

dataset_path = "dataset/QEvasion/train"
dataset = load_from_disk(dataset_path)

# Definiamo le 10 categorie del Task 2 (9 evasioni + Explicit)
system_prompt_task2 = """Sei un esperto analista di comunicazione politica. Il tuo compito è identificare la specifica tecnica di risposta utilizzata da un politico tra le seguenti 10 categorie:

1. 'Explicit': Risposta diretta e chiara che risponde esattamente alla domanda.
2. 'Declining to answer': Il politico rifiuta esplicitamente di rispondere.
3. 'Claims ignorance': Il politico dichiara di non avere le informazioni o di non sapere.
4. 'Dodging': Cambia leggermente argomento o risponde a una domanda diversa.
5. 'Deflection': Sposta la responsabilità su altri o attacca l'interlocutore.
6. 'General': Fornisce una risposta molto vaga o di carattere troppo generale.
7. 'Implicit': La risposta è sottintesa ma non dichiarata chiaramente.
8. 'Partial/half-answer': Risponde solo a una parte della domanda.
9. 'Clarification': Chiede chiarimenti sulla domanda per guadagnare tempo.
10. 'Other/Implicit Evasion': Altre forme di evasione non classificate.

Rispondi SOLO con il nome esatto della categoria, senza aggiungere altro testo."""

def format_prompt_task2(example):
    domanda = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    # Usiamo evasion_label invece di clarity_label
    label = example.get('evasion_label', 'Explicit') 
    
    user_message = f"Domanda: {domanda}\nRisposta del politico: {risposta}"
    
    messages = [
        {"role": "system", "content": system_prompt_task2},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": str(label)}
    ]
    
    testo_formattato = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": testo_formattato}

print("Formattazione dataset Task 2 in corso...")
formatted_dataset_task2 = dataset.map(
    format_prompt_task2, 
    remove_columns=dataset.column_names 
)

print("\n--- Esempio di Prompt Task 2 ---")
print(formatted_dataset_task2[0]["text"])

In [ ]:
# 6. TRAINING COMPLETO (Senza limiti di step)
import torch
import os
from transformers import TrainingArguments
from trl import SFTTrainer

output_model_dir = "./risultati_clarity_task2"
os.makedirs(output_model_dir, exist_ok=True)

training_arguments = TrainingArguments(
    output_dir=output_model_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_strategy="epoch",            # Salva un checkpoint alla fine di ogni epoca
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,                        # Sfrutta la tua GPU potente (A100/L4)
    max_grad_norm=0.3,
    num_train_epochs=3,               # Training completo su 3 epoche (consigliato)
    warmup_steps=100,                        
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none"                  # Evita di chiedere login a WandB
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_arguments,
    peft_config=None,                 # Il modello è già PeftModel
    processing_class=tokenizer
)

print("Inizio dell'addestramento completo! 🚀")
trainer.train()

# Salvataggio definitivo
final_save_path = f"{output_model_dir}/modello_finale"
trainer.model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"Modello salvato con successo in: {final_save_path}")

In [ ]:
# 7. Test e Inferenza - Task 2 (Evasion -> Clarity)
import torch

# 1. Definizione del Mapping Gerarchico (come da Paper)
mapping_gerarchico = {
    "Explicit": "Clear Reply",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Dodging": "Ambivalent Reply",
    "Deflection": "Ambivalent Reply",
    "General": "Ambivalent Reply",
    "Implicit": "Ambivalent Reply",
    "Partial/half-answer": "Ambivalent Reply",
    "Clarification": "Ambivalent Reply",
    "Other/Implicit Evasion": "Ambivalent Reply"
}

# 2. Funzione di Inferenza aggiornata
def predici_evasione(example):
    domanda = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    
    messages = [
        {"role": "system", "content": system_prompt_task2}, # Il prompt con le 10 classi
        {"role": "user", "content": f"Domanda: {domanda}\nRisposta del politico: {risposta}"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15, # Un po' di più per nomi di classi lunghi
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    predizione_evasione = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Mapping alla macro-categoria
    predizione_clarity = mapping_gerarchico.get(predizione_evasione, "Ambivalent Reply") # Default se sbaglia formato
    
    return predizione_evasione, predizione_clarity

# 3. Test rapido su 5 esempi
print("--- TEST QUALITATIVO TASK 2 ---")
for i in range(5):
    esempio = test_dataset[i]
    vero_task1 = esempio.get('clarity_label', '')
    vero_task2 = esempio.get('evasion_label', '')
    
    pred_evasione, pred_clarity = predici_evasione(esempio)
    
    print(f"\nEsempio {i+1}:")
    print(f"Target Reale (Task 2): {vero_task2}")
    print(f"PREDIZIONE LLM (Task 2): {pred_evasione}")
    print(f"Mapping Risultante (Task 1): {pred_clarity} (Vera: {vero_task1})")

In [ ]:
# 8. Valutazione Quantitativa Finale con Mapping
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_true_clarity = []
y_pred_clarity = []
y_pred_evasion_raw = [] # Per debug delle 10 classi

print(f"Valutazione su {len(test_dataset)} esempi...")

for i in tqdm(range(len(test_dataset))):
    esempio = test_dataset[i]
    y_true_clarity.append(str(esempio.get('clarity_label', '')).strip())
    
    evasione, clarity = predici_evasione(esempio)
    y_pred_clarity.append(clarity)
    y_pred_evasion_raw.append(evasione)

# REPORT FINALE (Quello da mettere nella tesi/progetto)
print("\n" + "="*20 + " REPORT FINALE TASK 2 (MAPPED TO TASK 1) " + "="*20)
print(classification_report(y_true_clarity, y_pred_clarity))

# Visualizzazione della distribuzione delle 10 classi predette
plt.figure(figsize=(10, 4))
pd.Series(y_pred_evasion_raw).value_counts().plot(kind='bar')
plt.title("Distribuzione delle tecniche di evasione predette")
plt.show()

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import json

# 1. Estrazione dei log dall'oggetto trainer
# trainer.state.log_history contiene una lista di dizionari con i log registrati
history = trainer.state.log_history

# 2. Trasformazione in un DataFrame di Pandas per manipolarli facilmente
df_logs = pd.DataFrame(history)

# 3. Pulizia dei dati
# I log contengono sia i dati di training che (eventualmente) quelli di eval e riepilogo finale.
# Teniamo solo le colonne interessanti e rimuoviamo le righe vuote per la loss.
columns_to_keep = ['step', 'loss', 'learning_rate', 'epoch', 'eval_loss', 'eval_accuracy']
df_filtered = df_logs[[c for c in columns_to_keep if c in df_logs.columns]].dropna(subset=['loss'], how='all')

# 4. Salvataggio su Google Drive
csv_filename = f"{output_model_dir}/training_logs.csv"
json_filename = f"{output_model_dir}/training_logs_full.json"

df_filtered.to_csv(csv_filename, index=False)

# Salviamo anche la versione JSON completa (che contiene metadati extra)
with open(json_filename, "w") as f:
    json.dump(history, f, indent=4)

print(f"Dati della loss salvati con successo!")
print(f"CSV creato: {csv_filename}")
print(f"JSON completo creato: {json_filename}")

# Visualizziamo le ultime righe del log per conferma
print("\nUltimi log registrati:")
print(df_filtered.tail())